# 03 — Design, power e aprovação
O registro só vira `approved` depois que **o mesmo `DesignSpec`** passa pelo power
gate e por A/A do procedimento completo. Para seleção observacional de mercados,
o callback de A/A deve reproduzir seleção, exclusões, estimadores e regra de decisão;
um A/A com cidades sorteadas não autoriza o desenho selecionado.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-2.0.0a1-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from dataclasses import replace
from datetime import date, timedelta
import json, uuid

from supply_experiments.calibration.aa import run_aa_calibration
from supply_experiments.design.power import power_analysis
from supply_experiments.design.spec import DecisionRule, DesignSpec
from supply_experiments.design.treated_selection import (recommend_treated_sets,
                                                         recommendations_frame)
from supply_experiments.estimators import fit_ascm, fit_scm, fit_sdid
from supply_experiments.io import (TABLES, ExperimentRecord, active_blocked_cities,
                                   ensure_registry, load_city_panel, save_experiment)

POST_DAYS, PRE_DAYS = 35, 122
ALPHA, MAX_PLACEBOS = 0.10, 30
EFFECT_GRID = (0.0, 0.02, 0.03, 0.05, 0.08)
FIT_FNS = {"scm": fit_scm, "ascm": fit_ascm, "sdid": fit_sdid}
RULE = DecisionRule(min_rejections=2, direction="positive",
                    methods=("scm", "ascm", "sdid"))
SELECTION_PROCEDURE_ID = "recommend-treated-sets-v1"
SELECTOR_CONFIG = {
    "selector": "recommend_treated_sets",
    "n_treated": [2, 3],
    "max_gmv_share_per_city": 0.05,
    "min_donors": 10,
    "top_k_cities": 12,
    "screen_donor_cap": 40,
    "spillover_radius_km": 40.0,
}

tables = TABLES["sandbox"]
panel, stats = load_city_panel(spark, "2025-06-01", "2026-06-30")
blocked = active_blocked_cities(spark, tables["registry"])
candidates = [c for c in stats.index if c in panel.cities
              and c not in blocked["treated_active"] | blocked["control_active"]]

## 1. Propose a design without post-treatment data
The ranking is a selection algorithm, not random assignment. Its exact deployed
version must therefore be replayed in A/A below.

In [ ]:
coords = {}  # supply coordinates/adjacency in production; empty means no radius screen
recs = recommend_treated_sets(
    panel, stats, candidates, fit_scm,
    pre_days=PRE_DAYS, post_days=POST_DAYS, n_treated=(2, 3),
    coords=coords, max_gmv_share_per_city=0.05,
)
display(recommendations_frame(recs))

In [ ]:
CHOSEN_RANK = 1
TREATED = tuple(recs[CHOSEN_RANK - 1].cities)
DONORS = tuple(recs[CHOSEN_RANK - 1].donors[:20])
HYPOTHESIS = "State the directional causal hypothesis and mechanism here"
EXPECTED_EFFECT = 0.05
EXPERIMENT_ID = str(uuid.uuid4())
START = date.today() + timedelta(days=7)
END = START + timedelta(days=POST_DAYS - 1)

draft_spec = DesignSpec(
    experiment_id=EXPERIMENT_ID, treated_units=TREATED, donor_units=DONORS,
    estimator_names=tuple(FIT_FNS), pre_days=PRE_DAYS, post_days=POST_DAYS,
    start_date=START, end_date=END, alpha=ALPHA,
    max_group_placebos=MAX_PLACEBOS, effect_grid=EFFECT_GRID,
    decision_rule=RULE, primary_kpi="gmv",
    assignment_mechanism="observational",
    selection_procedure_id=SELECTION_PROCEDURE_ID,
    eligible_units=tuple(candidates),
    selector_config_json=json.dumps(SELECTOR_CONFIG),
    max_pre_rmspe=0.10, require_calibration=True, calibration_scope="exact_design",
)
print("Draft design fingerprint:", draft_spec.fingerprint)

SELECTION_REPLAY_IMPLEMENTED = False
def replay_deployed_selection(panel_for_run, eligible, rng, run, start_idx):
    # Re-run SELECTION_PROCEDURE_ID using only information available before start_idx.
    # It must return exactly (n_treated, n_donors) cities after the same exclusions.
    # The version ID is the code-identity boundary: keep that implementation and
    # SELECTOR_CONFIG versioned together because Python callback bytecode is not hashed.
    raise NotImplementedError("Implement the deployed selector replay before approval")

assert SELECTION_REPLAY_IMPLEMENTED, 'selection replay is required for power and A/A'

## 2. Power the executable rule
This runs all three estimators and the registered two-of-three rule. Invalid fits and
the pre-fit gate are reported separately; they are not counted as non-rejections.

In [ ]:
pw = power_analysis(
    panel, TREATED, DONORS, fit_scm, pre_days=PRE_DAYS, post_days=POST_DAYS,
    effect_grid=EFFECT_GRID, n_sims_per_point=30, alpha=ALPHA,
    max_group_placebos=MAX_PLACEBOS, fit_fns=FIT_FNS,
    decision_rule=RULE, design_spec=draft_spec,
    selection_fn=replay_deployed_selection, eligible=candidates,
    selection_procedure_id=SELECTION_PROCEDURE_ID,
)
for effect in pw.effect_grid:
    print(effect, pw.power[effect], pw.power_ci[effect], pw.invalid_rate[effect])
assert pw.design_fingerprint == draft_spec.fingerprint
assert pw.mde_80 is not None and pw.mde_80 <= EXPECTED_EFFECT

## 3. Replay market selection under the null
Implement this callback from the versioned production selector. Returning the already
chosen cities is **not** a replay when outcomes influenced their selection. The explicit
assertion prevents accidental approval of the placeholder.

In [ ]:
assert pw.selection_scope == "replayed_selection"
assert pw.selection_procedure_id == SELECTION_PROCEDURE_ID

In [ ]:
aa = run_aa_calibration(
    panel, candidates, fit_scm, pre_days=PRE_DAYS, post_days=POST_DAYS,
    n_runs=200, n_treated=len(TREATED), n_donors=len(DONORS), alpha=ALPHA,
    max_group_placebos=MAX_PLACEBOS,
    min_valid_runs=draft_spec.calibration_min_valid_runs,
    min_ks_p_value=draft_spec.calibration_min_ks_p_value,
    max_fpr_inflation=draft_spec.calibration_max_fpr_inflation,
    max_invalid_rate=draft_spec.calibration_max_invalid_rate,
    fit_fns=FIT_FNS, decision_rule=RULE,
    selection_fn=replay_deployed_selection,
    selection_procedure_id=SELECTION_PROCEDURE_ID,
    design_fingerprint=draft_spec.fingerprint,
    assignment_mechanism=draft_spec.assignment_mechanism,
    max_pre_rmspe=draft_spec.max_pre_rmspe,
    seed=draft_spec.calibration_seed,
    design_spec=draft_spec,
)
assert aa.passed, aa.failure_reasons
spec = replace(draft_spec, calibration_fingerprint=aa.calibration_fingerprint)
assert aa.matches_design(spec)
from datetime import datetime
spark.createDataFrame([{
    "run_at": datetime.utcnow(),
    "calibration_fingerprint": aa.calibration_fingerprint,
    "design_fingerprint": aa.design_fingerprint,
    "passed": aa.passed,
    "selection_scope": aa.selection_scope,
    "procedure_scope": aa.procedure_scope,
    "artifact_json": aa.to_json(include_details=False),
}]).write.format("delta").mode("append").option("mergeSchema", "true")   .saveAsTable(tables["calibration"])
print("Calibration fingerprint:", aa.calibration_fingerprint)

## 4. Approve the immutable bound record
`save_experiment` rejects an unbound design, a failed power gate, a missing calibration
digest, late approval, or a mutation after approval.

In [ ]:
rec = ExperimentRecord(
    experiment_id=EXPERIMENT_ID, name="geo_experiment",
    hypothesis=HYPOTHESIS, status="approved", design_method="ascm",
    primary_kpi=spec.primary_kpi, guardrail_kpis=["rupture_rate_order"],
    start_date=START, end_date=END, pre_window_days=PRE_DAYS,
    treated_units=[{"city_norm": c, "city_address": c, "state_address": "",
                    "weight": None} for c in TREATED],
    control_units=[{"city_norm": c, "city_address": c,
                    "state_address": str(stats.loc[c, "state_address"]),
                    "weight": None} for c in DONORS],
    mde_estimated=pw.mde_80, expected_effect=EXPECTED_EFFECT,
    decision_rule=json.dumps(RULE.to_dict(), sort_keys=True),
    created_by=spark.sql("SELECT current_user()").first()[0],
).with_design_contract(spec, aa.calibration_fingerprint, power_result=pw)
ensure_registry(spark, tables["registry"])
save_experiment(spark, tables["registry"], rec, alpha=ALPHA)
print("Approved experiment:", rec.experiment_id)